In [13]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
%pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-1e1891a5-92b8-49c3-77ee-7ff04727ceae, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [14]:
GITHUB_USERNAME = "ros1nant"

assert GITHUB_USERNAME.strip(), "Enter your GitHub username."
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: ros1nant


In [17]:
%%writefile distributed_cuda_solver.py
"""Four-rank anisotropic reaction-diffusion reference solver for Google Colab."""
import hashlib
import json
import math
import re
import subprocess
import sys
import traceback
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = GLOBAL_NY = 512
NY = NX = 256
STEPS = 400
LAMBDA_DAMP = 0.15
UP, DOWN, LEFT, RIGHT = range(4)
OPPOSITE = (DOWN, UP, RIGHT, LEFT)
MANIFEST_PATH = Path("submission_manifest.json")


def get_gpu_uuid():
    output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        text=True,
    )
    return output.splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8"))
    return int(digest.hexdigest()[:16], 16)


def init_distributed_context(github_user):
    world = MPI.COMM_WORLD
    if world.Get_size() != 4:
        raise RuntimeError(f"Expected 4 MPI ranks, got {world.Get_size()}")
    cart = world.Create_cart(dims=(2, 2), periods=(False, False), reorder=False)
    rank = cart.Get_rank()
    coords = cart.Get_coords(rank)
    neighbors = (*cart.Shift(0, 1), *cart.Shift(1, 1))
    cuda.select_device(0)
    gpu_uuid = cart.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    seed = derive_student_seed(github_user, gpu_uuid)
    return cart, coords, neighbors, gpu_uuid, seed


def initialize_subdomains(coords, neighbors, seed):
    row_rank, col_rank = coords
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    x = (col_rank * NX + np.arange(NX + 2, dtype=np.float64))[None, :] * dx
    y = (row_rank * NY + np.arange(NY + 2, dtype=np.float64))[:, None] * dy
    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y)

    rng = np.random.default_rng(seed)
    initial = np.zeros((NY + 2, NX + 2), dtype=np.float64)
    for _ in range(3):
        center_x, center_y = rng.uniform(0.37, 0.63, size=2)
        amplitude = rng.uniform(0.65, 1.0)
        radius = 0.15
        initial += amplitude * np.exp(
            -((x - center_x) ** 2 + (y - center_y) ** 2) / (2.0 * radius**2)
        )
    source = 0.005 * np.exp(-((x - 0.50) ** 2 + (y - 0.55) ** 2) / (2.0 * 0.07**2))

    if neighbors[UP] == MPI.PROC_NULL:
        initial[0, :] = 0.0
    if neighbors[DOWN] == MPI.PROC_NULL:
        initial[-1, :] = 0.0
    if neighbors[LEFT] == MPI.PROC_NULL:
        initial[:, 0] = 0.0
    if neighbors[RIGHT] == MPI.PROC_NULL:
        initial[:, -1] = 0.0
    return initial, kappa, source, dx, dy


@cuda.jit
def pack_halos_kernel(u, up, down, left, right):
    lane = cuda.grid(1)
    ny, nx = u.shape[0] - 2, u.shape[1] - 2
    if lane < nx:
        up[lane] = u[1, lane + 1]
        down[lane] = u[ny, lane + 1]
    if lane < ny:
        left[lane] = u[lane + 1, 1]
        right[lane] = u[lane + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, up, down, left, right):
    lane = cuda.grid(1)
    ny, nx = u.shape[0] - 2, u.shape[1] - 2
    if lane < nx:
        u[0, lane + 1] = up[lane]
        u[ny + 1, lane + 1] = down[lane]
    if lane < ny:
        u[lane + 1, 0] = left[lane]
        u[lane + 1, nx + 1] = right[lane]


@dataclass
class HaloBuffers:
    send_host: tuple
    recv_host: tuple
    send_device: tuple
    recv_device: tuple


def allocate_halos():
    lengths = (NX, NX, NY, NY)
    send_host = tuple(cuda.pinned_array(n, dtype=np.float64) for n in lengths)
    recv_host = tuple(cuda.pinned_array(n, dtype=np.float64) for n in lengths)
    for buffer in recv_host:
        # A receive from MPI.PROC_NULL leaves its buffer unchanged.
        buffer.fill(0.0)
    send_device = tuple(cuda.device_array(n, dtype=np.float64) for n in lengths)
    recv_device = tuple(cuda.device_array(n, dtype=np.float64) for n in lengths)
    return HaloBuffers(send_host, recv_host, send_device, recv_device)


def exchange_halos_async(cart, neighbors, u, buffers, stream_boundary, start, end):
    """Stage outgoing halos and post receives before sends. Return pending requests."""
    blocks = (max(NX, NY) + 127) // 128
    start.record(stream_boundary)
    pack_halos_kernel[blocks, 128, stream_boundary](u, *buffers.send_device)
    for device, host, neighbor in zip(buffers.send_device, buffers.send_host, neighbors):
        if neighbor != MPI.PROC_NULL:
            device.copy_to_host(host, stream=stream_boundary)
    end.record(stream_boundary)
    stream_boundary.synchronize()

    requests = [
        cart.Irecv(buffers.recv_host[d], source=neighbor, tag=OPPOSITE[d])
        for d, neighbor in enumerate(neighbors)
    ]
    requests.extend(
        cart.Isend(buffers.send_host[d], dest=neighbor, tag=d)
        for d, neighbor in enumerate(neighbors)
    )
    return requests


def receive_halos(u, buffers, stream_boundary):
    for device, host in zip(buffers.recv_device, buffers.recv_host):
        device.copy_to_device(host, stream=stream_boundary)
    blocks = (max(NX, NY) + 127) // 128
    unpack_halos_kernel[blocks, 128, stream_boundary](u, *buffers.recv_device)


@cuda.jit
def tiled_stencil_kernel(u, out, kappa, source, dt, inv_dx2, inv_dy2,
                         r_start, r_end, c_start, c_end):
    """Update a box with exclusive end indices."""
    smem = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    sx, sy = tx + 1, ty + 1
    r = r_start + cuda.blockIdx.y * TILE_DIM + ty
    c = c_start + cuda.blockIdx.x * TILE_DIM + tx
    active = r < r_end and c < c_end

    if active:
        smem[sy, sx] = u[r, c]
        if tx == 0:
            smem[sy, sx - 1] = u[r, c - 1]
        if tx == TILE_DIM - 1 or c == c_end - 1:
            smem[sy, sx + 1] = u[r, c + 1]
        if ty == 0:
            smem[sy - 1, sx] = u[r - 1, c]
        if ty == TILE_DIM - 1 or r == r_end - 1:
            smem[sy + 1, sx] = u[r + 1, c]

    cuda.syncthreads()

    if active:
        center = smem[sy, sx]
        k_center = kappa[r, c]
        k_east = 0.5 * (k_center + kappa[r, c + 1])
        k_west = 0.5 * (k_center + kappa[r, c - 1])
        k_south = 0.5 * (k_center + kappa[r + 1, c])
        k_north = 0.5 * (k_center + kappa[r - 1, c])
        flux_x = (k_east * (smem[sy, sx + 1] - center)
                  - k_west * (center - smem[sy, sx - 1])) * inv_dx2
        flux_y = (k_south * (smem[sy + 1, sx] - center)
                  - k_north * (center - smem[sy - 1, sx])) * inv_dy2
        damping = LAMBDA_DAMP * center * center * center
        out[r, c] = center + dt * (flux_x + flux_y - damping + source[r, c])


def launch_box(stencil_args, box, stream):
    r_start, r_end, c_start, c_end = box
    blocks = (
        (c_end - c_start + TILE_DIM - 1) // TILE_DIM,
        (r_end - r_start + TILE_DIM - 1) // TILE_DIM,
    )
    tiled_stencil_kernel[blocks, (TILE_DIM, TILE_DIM), stream](*stencil_args, *box)


@dataclass
class StepEvents:
    pack_start: object
    pack_end: object
    interior_start: object
    interior_end: object
    ring_start: object
    ring_end: object


def launch_interior(stencil_args, stream_interior, events):
    events.interior_start.record(stream_interior)
    launch_box(stencil_args, (2, NY, 2, NX), stream_interior)
    events.interior_end.record(stream_interior)


def finish_boundary(stencil_args, buffers, stream_boundary, events):
    events.ring_start.record(stream_boundary)
    receive_halos(stencil_args[0], buffers, stream_boundary)
    boxes = (
        (1, 2, 1, NX + 1),
        (NY, NY + 1, 1, NX + 1),
        (2, NY, 1, 2),
        (2, NY, NX, NX + 1),
    )
    for box in boxes:
        launch_box(stencil_args, box, stream_boundary)
    events.ring_end.record(stream_boundary)


def step_overlapped(cart, neighbors, stencil_args, buffers, stream_boundary,
                    stream_interior, events):
    pending = exchange_halos_async(
        cart, neighbors, stencil_args[0], buffers, stream_boundary,
        events.pack_start, events.pack_end,
    )
    launch_interior(stencil_args, stream_interior, events)
    MPI.Request.Waitall(pending)
    finish_boundary(stencil_args, buffers, stream_boundary, events)
    # Complete both writers before swapping the current and next fields.
    stream_boundary.synchronize()
    stream_interior.synchronize()


def step_serialized(cart, neighbors, stencil_args, buffers, stream_boundary,
                    stream_interior, events):
    pending = exchange_halos_async(
        cart, neighbors, stencil_args[0], buffers, stream_boundary,
        events.pack_start, events.pack_end,
    )
    MPI.Request.Waitall(pending)
    finish_boundary(stencil_args, buffers, stream_boundary, events)
    stream_boundary.synchronize()
    launch_interior(stencil_args, stream_interior, events)
    stream_interior.synchronize()


def record_diagnostics(cart, current, previous, step, dx, dy):
    owned = current.copy_to_host()[1:-1, 1:-1]
    residual_sq = 0.0
    if previous is not None:
        old_owned = previous.copy_to_host()[1:-1, 1:-1]
        diff = owned - old_owned
        residual_sq = np.sum(diff * diff, dtype=np.float64)
    local = np.array([np.sum(owned, dtype=np.float64) * dx * dy, residual_sq])
    total = np.empty_like(local)
    cart.Allreduce(local, total, op=MPI.SUM)
    entry = {"step": step, "energy": float(total[0]), "l2_residual": math.sqrt(total[1])}
    if cart.Get_rank() == 0:
        print(f"step={step:3d}  energy={entry['energy']:.16e}  "
              f"L2 residual={entry['l2_residual']:.16e}", flush=True)
    return entry


def benchmark_mode(cart, neighbors, initial, material_args, buffers,
                   stream_boundary, stream_interior, dx, dy, step_function):
    current = cuda.to_device(initial)
    next_field = cuda.to_device(initial)
    events = StepEvents(*(cuda.event(timing=True) for _ in range(6)))
    if cart.Get_rank() == 0:
        print(f"\nMode {step_function.__name__}", flush=True)
    history = [record_diagnostics(cart, current, None, 0, dx, dy)]
    interior_ms = 0.0
    boundary_ms = 0.0
    cart.Barrier()
    started = MPI.Wtime()
    for step in range(1, STEPS + 1):
        stencil_args = (current, next_field, *material_args)
        step_function(cart, neighbors, stencil_args, buffers,
                      stream_boundary, stream_interior, events)
        interior_ms += cuda.event_elapsed_time(events.interior_start, events.interior_end)
        boundary_ms += cuda.event_elapsed_time(events.pack_start, events.pack_end)
        boundary_ms += cuda.event_elapsed_time(events.ring_start, events.ring_end)
        previous, current, next_field = current, next_field, current
        if step % 100 == 0:
            history.append(record_diagnostics(cart, current, previous, step, dx, dy))
    wall_ms = (MPI.Wtime() - started) * 1000.0
    maxima = np.empty(3, dtype=np.float64)
    cart.Allreduce(np.array([wall_ms, interior_ms, boundary_ms]),
                   maxima, op=MPI.MAX)
    timings = {
        "wall_ms": float(maxima[0]),
        "interior_cuda_ms": float(maxima[1]),
        "boundary_cuda_ms": float(maxima[2]),
    }
    return current.copy_to_host()[1:-1, 1:-1].copy(), history, timings


def audit_ptx():
    assembly = tiled_stencil_kernel.inspect_asm()
    if len(assembly) != 1:
        raise RuntimeError(f"Expected one stencil specialization, found {len(assembly)}")
    ptx = next(iter(assembly.values()))
    counts = {}
    for mnemonic in ("st.shared.f64", "ld.shared.f64", "bar.sync"):
        pattern = rf"(?m)^\s*(?:@!?%\w+\s+)?{re.escape(mnemonic)}(?=\s)"
        counts[mnemonic] = len(re.findall(pattern, ptx))
    return counts, all(counts.values())


def run_solver(github_user):
    cart, coords, neighbors, gpu_uuid, seed = init_distributed_context(github_user)
    rank = cart.Get_rank()
    if rank == 0:
        MANIFEST_PATH.unlink(missing_ok=True)
        print(f"GITHUB_USERNAME={github_user}  GPU_UUID={gpu_uuid}  INSTANCE_SEED={seed}", flush=True)
        print(f"mpi_ranks={cart.Get_size()}  cart_dims=2x2  MPI.PROC_NULL={MPI.PROC_NULL}", flush=True)

    initial, kappa, source, dx, dy = initialize_subdomains(coords, neighbors, seed)
    # Include physical edge values because they participate in face conductivities.
    kappa_max = cart.allreduce(float(kappa.max()), op=MPI.MAX)
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = 0.8 * dt_max
    print(f"rank={rank}  coords={tuple(coords)}  neighbors (U,D,L,R)={neighbors}  "
          f"local_shape={initial.shape}", flush=True)
    if rank == 0:
        packed = NY * initial.itemsize
        bounding = ((NY - 1) * initial.shape[1] + 1) * initial.itemsize
        print(f"column stride={initial.strides[0]} B  packed={packed} B  "
              f"bounding={bounding} B  extra={bounding - packed} B per column", flush=True)
        print(f"kappa_max={kappa_max:.16e}  dx={dx:.16e}  "
              f"dt_max={dt_max:.16e}  dt={dt:.16e}", flush=True)
        print("Wall time includes 400 steps with event collection and diagnostics. "
              "CUDA event times sum per-stream intervals.", flush=True)

    material_args = (cuda.to_device(kappa), cuda.to_device(source),
                     dt, 1.0 / dx**2, 1.0 / dy**2)
    buffers = allocate_halos()
    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    # Compile all kernels and initialize both streams before the timed modes.
    warm_args = (cuda.to_device(initial), cuda.to_device(initial), *material_args)
    warm_events = StepEvents(*(cuda.event(timing=True) for _ in range(6)))
    step_overlapped(cart, neighbors, warm_args, buffers, stream_boundary,
                    stream_interior, warm_events)
    del warm_args, warm_events

    mode_args = (cart, neighbors, initial, material_args, buffers,
                 stream_boundary, stream_interior, dx, dy)
    serial, serial_history, serial_times = benchmark_mode(*mode_args, step_serialized)
    overlap, overlap_history, overlap_times = benchmark_mode(*mode_args, step_overlapped)
    finite = cart.allreduce(bool(np.isfinite(serial).all() and np.isfinite(overlap).all()),
                            op=MPI.LAND)
    local_max = float(np.max(np.abs(serial - overlap))) if finite else math.inf
    parity_max_diff = cart.allreduce(local_max, op=MPI.MAX)

    if rank == 0:
        ptx_counts, ptx_passed = audit_ptx()
        speedup = serial_times["wall_ms"] / overlap_times["wall_ms"]
        print(f"PTX counts: {ptx_counts}  ptx_audit_passed={ptx_passed}", flush=True)
        print(f"serialized_ms={serial_times['wall_ms']:.3f}  "
              f"overlapped_ms={overlap_times['wall_ms']:.3f}  speedup={speedup:.5f}x", flush=True)
        print(f"CUDA events (max rank, ms): serialized={serial_times}  "
              f"overlapped={overlap_times}", flush=True)
        print(f"parity_max_diff={parity_max_diff:.16e}", flush=True)
        if not ptx_passed or not parity_max_diff < 1e-12:
            raise RuntimeError("Verification failed: see the PTX counts and parity above.")

        manifest = {
            "github_username": github_user,
            "gpu_uuid": gpu_uuid,
            "instance_seed": seed,
            "mpi_ranks": cart.Get_size(),
            "cart_dims": [2, 2],
            "steps": STEPS,
            "kappa_max": kappa_max,
            "dx": dx,
            "dy": dy,
            "dt": dt,
            "dt_max": dt_max,
            "ptx_instruction_counts": ptx_counts,
            "ptx_audit_passed": ptx_passed,
            "parity_max_diff": parity_max_diff,
            "serialized_ms": serial_times["wall_ms"],
            "overlapped_ms": overlap_times["wall_ms"],
            "cuda_event_timings_ms": {
                mode: {key: value for key, value in timings.items() if key != "wall_ms"}
                for mode, timings in (("serialized", serial_times), ("overlapped", overlap_times))
            },
            "energy_and_residual": {"serialized": serial_history, "overlapped": overlap_history},
        }
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n",
                                 encoding="utf-8")
        print(f"Wrote {MANIFEST_PATH}", flush=True)
    cart.Barrier()
    cart.Free()


if __name__ == "__main__":
    try:
        if (len(sys.argv) != 2 or not sys.argv[1].strip()
                or sys.argv[1] == "REPLACE_WITH_YOUR_GITHUB_USERNAME"):
            raise SystemExit("Usage: distributed_cuda_solver.py YOUR_GITHUB_USERNAME")
        run_solver(sys.argv[1].strip())
    except Exception:
        traceback.print_exc()
        MPI.COMM_WORLD.Abort(1)


Overwriting distributed_cuda_solver.py


In [18]:
import subprocess
import sys

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    sys.executable,
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout, end="")

if proc.stderr:
    print(proc.stderr, end="")

proc.check_returncode()

GITHUB_USERNAME=ros1nant  GPU_UUID=GPU-1e1891a5-92b8-49c3-77ee-7ff04727ceae  INSTANCE_SEED=16467635541297678563
mpi_ranks=4  cart_dims=2x2  MPI.PROC_NULL=-2
rank=2  coords=(1, 0)  neighbors (U,D,L,R)=(0, -2, -2, 3)  local_shape=(258, 258)
rank=0  coords=(0, 0)  neighbors (U,D,L,R)=(-2, 2, -2, 1)  local_shape=(258, 258)
column stride=2064 B  packed=2048 B  bounding=526328 B  extra=524280 B per column
kappa_max=2.3999962497115201e-01  dx=1.9493177387914229e-03  dt_max=3.9581724838358046e-06  dt=3.1665379870686439e-06
Wall time includes 400 steps with event collection and diagnostics. CUDA event times sum per-stream intervals.
rank=1  coords=(0, 1)  neighbors (U,D,L,R)=(-2, 3, 0, -2)  local_shape=(258, 258)
rank=3  coords=(1, 1)  neighbors (U,D,L,R)=(1, -2, 2, -2)  local_shape=(258, 258)

Mode step_serialized
step=  0  energy=3.5862080176832067e-01  L2 residual=0.0000000000000000e+00
step=100  energy=3.5827558839647500e-01  L2 residual=8.9814646993159591e-03
step=200  energy=3.58077487594

Q1. Memory coalescing and strided column packing

The run reported a local shape of 258 × 258 and a column stride of 2,064 bytes. In a C-contiguous array of doubles, the byte offset is 8 × (258 × row + column). Successive lanes reading a row access adjacent eight-byte values, so the warp can combine their requests into a small number of memory transactions. Successive lanes reading a halo column touch addresses 2,064 bytes apart, so their reads are scattered across memory segments. A single contiguous DMA copy from the column's first element to its last would also transfer the intervening values from other columns.

For one 256-element column, the contiguous span is (255 × 258 + 1) × 8 = 526,328 bytes. The packed buffer is 256 × 8 = 2,048 bytes, saving 524,280 bytes per column per copy. If both columns were copied in one direction, the two bounding spans would transfer 1,048,560 extra bytes in total. The pack kernel still makes strided global-memory reads from the original column, but writes contiguous outputs that can be copied efficiently to pinned host memory.


Q2. Shared-memory tile indexing and bank conflicts

The run used an 18 × 18 float64 tile with a 16 × 16 block, and the compiled kernel reported five st.shared.f64 and five ld.shared.f64 instruction sites. In a full tile, each active thread places its center at smem[ty + 1, tx + 1]. The two corner threads perform these accesses:
Thread (tx, ty)	Writes before the barrier	Reads after the barrier, in center / west / east / north / south order
(0, 0)	center [1,1], west apron [1,0], north apron [0,1]	[1,1], [1,0], [1,2], [0,1], [2,1]
(15, 15)	center [16,16], east apron [16,17], south apron [17,16]	[16,16], [16,15], [16,17], [15,16], [17,16]


With 32 four-byte banks, each double occupies two adjacent banks. The starting bank is (base_bank + 2 × (18 × sy + sx)) mod 32. A group of 16 lanes reading consecutive doubles along a row uses all 32 banks once. A full warp's 256-byte request needs two 128-byte phases for the access width, and row-wise accesses introduce no additional 2-way conflict. With lanes instead moving down a column at fixed sx, each next address advances 36 bank words, or 4 banks modulo 32. Bank pairs repeat every eight lanes, so a 16-lane phase encounters a 2-way bank conflict on distinct addresses.

The kernel maps consecutive lanes across tx, so each warp spans two adjacent rows. Even for a north or south stencil load, each 16-lane group reads contiguous addresses within its target row. The column-stride conflict describes a hypothetical column-mapped warp. The PTX counts are static and do not directly measure bank conflicts. NVIDIA CUDA Programming Guide

Q3. PTX audit and synchronization

My run reported 5 st.shared.f64, 5 ld.shared.f64, and 1 bar.sync in tiled_stencil_kernel.inspect_asm(). The audit passed. These are static instruction counts; branches determine which stores each thread executes.

A 16 × 16 block has 256 threads, or eight warps. For example, thread (tx, ty) = (0, 1) is in warp 0, but its south neighbor smem[3,1] is written by thread (0, 2) in warp 1. Without a block-wide barrier, warp 0 could read that location before warp 1 writes it. This is a read-after-write hazard and can feed an uninitialized or stale value into the stencil. The barrier guarantees that the tile and apron writes are visible before any thread reads its stencil neighbors. All threads reach the barrier, including inactive threads in partial tiles.

Q4. Four CUDA contexts on one T4

Metric	Serialized	Overlapped
Wall time for 400 steps	2,093.286 ms	1,995.955 ms
Interior CUDA event sum, max across ranks	92.100 ms	205.060 ms
Boundary CUDA event sum, max across ranks	952.967 ms	895.006 ms


Overlap saved 97.331 ms, a 4.65% wall-time reduction and a 1.04876× speedup. The final maximum absolute state difference was 0, so both modes produced the same result in this run.

Each MPI process has a CUDA context on device 0. With ordinary context scheduling and no MPS, contexts share the GPU through time slicing. The SMs, copy engines, and PCIe bandwidth are shared physical resources. Within a context, pinned-buffer DMA and kernel work can overlap when stream dependencies and hardware scheduling permit it. In the overlapped mode, the host posts MPI requests, queues the deep-interior stencil, and then waits for the halos. That lets some useful GPU work run during MPI progress. The larger interior event sum can include scheduling and resource contention; it does not by itself show that the stencil performed more arithmetic. Event intervals may overlap, and each maximum may come from a different rank, so these sums cannot be added to reconstruct wall time.

With one GPU per rank, each rank has its own SMs and copy engines. Across InfiniBand, network transfers can progress while the dedicated GPU works on its interior. Host staging, PCIe bandwidth, and MPI progress still affect the result, and GPUDirect RDMA would only apply if supported and configured. NVIDIA CUDA Best Practices Guide

Q5. Blocking send deadlock and MPI.PROC_NULL

Each halo message in my run contains 256 × 8 = 2,048 bytes. If a message fits the MPI transport's eager limit and buffer space is available, MPI may copy it into internal storage and let a blocking standard-mode Send return before the peer posts its Recv. With a larger halo, a transport can switch to rendezvous: the send waits for the matching receive to participate before the transfer can finish. The threshold depends on the MPI implementation, transport, configuration, and available buffer space. The run did not measure the transport's eager limit.

For example, rank 0's down neighbor is rank 2, and rank 2's up neighbor is rank 0. If both ranks reach a blocking Send to each other before either posts its receive and neither message can be buffered, each waits for a receive the other rank cannot yet post. My implementation posts all Irecv requests first, then all Isend requests, and only then calls Waitall.

Rank 0 at (0,0) reported neighbors (-2, 2, -2, 1) in up, down, left, right order. Here -2 is MPI.PROC_NULL. A send to it completes without moving data, and a receive from it completes without modifying the receive buffer. The up and left receive buffers were initialized to zero and remain zero, so unpacking them maintains the homogeneous Dirichlet boundary while using the same communication pattern as the other ranks. MPI 4.1 Standard:

Q6. CFL limit and measured thermal energy

My run reported kappa_max = 0.23999962497115201 and dx = dy = 1/513 = 0.001949317738791423. The two-dimensional explicit diffusion bound is
Delta t_{max} = rac{(1/513)^2}{4(0.23999962497115201)}
approx 3.958172483835805 	imes 10^{-6}.
The simulation used dt = 3.166537987068644e-6, which is 80% of the bound. The diffusion update assigns nonnegative weights to the four neighbors. The sum of their weights is at most 4 × kappa_max × dt / dx² = 0.8, because each face diffusivity is an average bounded by kappa_max. Thus the center weight for the diffusion update is at least 0.2. This bound controls the diffusion term; the source and cubic damping also contribute to the observed evolution.

Both modes printed the same Allreduce energy and L2 residual values:
Step	Thermal energy	L2 residual
0	0.35862080176832067	0 (initial state)
100	0.35827558839647500	0.008981464699315959
200	0.35807748759497771	0.008413906025055229
300	0.35790784710221424	0.008229913583183675
400	0.35775234128010297	0.008122209201571489


The total dropped by 0.00086846, or 0.2422%, and declined at every recorded checkpoint. Summing the finite-volume-form stencil over all owned cells cancels each internal face flux with its equal and opposite counterpart, including faces between MPI ranks. The remaining energy change comes from flux through the zero-temperature physical boundary, cubic damping, and the localized source. In this run the losses outweighed the source across each sampled interval. The checkpoint data demonstrate measured dissipation at the sampled steps.